In [1]:
import cv2
import time
import math
import json
import os
import csv
import numpy as np
import mediapipe as mp
from collections import deque, defaultdict

# CONFIG

In [2]:
TARGET_FPS = 8
PROCESS_INTERVAL = 1.0 / TARGET_FPS

FACING_YAW_DEG = 20.0
FACING_PITCH_DEG = 15.0
MIN_FACING_FOR_GAZE = 12.0

HYSTERESIS_FRAMES = 6
EMA_ALPHA = 0.4

IOU_THRESH = 0.3
MAX_MISSED_SEC = 2.0

CHEAT_MIN_OFFPOSE_SEC = 1.2
CHEAT_MIN_MULTIFACE_SEC = 1.0
CHEAT_MIN_OUTOFFRAME_SEC = 1.0
CHEAT_MIN_EYESOFF_SEC = 1.0

CALIB_SECONDS = 2.0
GAZE_DELTA = 0.18

ENABLE_CSV_LOG = False
CSV_PATH = "cheat_events.csv"
SHOW_DEBUG = True

LMK_IDX = [33, 263, 1, 61, 291, 199]
GAZE_LEFT_LMK = [468, 469, 470, 471]
GAZE_RIGHT_LMK = [472, 473, 474, 475]
GAZE_RIGHT_LMK_VARIANT_B = [473, 474, 475, 476]
EYE_LEFT_CORNERS = (33, 133)
EYE_RIGHT_CORNERS = (263, 362)

# UTILS

In [3]:
def now_ts(): return time.time()

def iou(a, b):
    xA, yA = max(a[0], b[0]), max(a[1], b[1])
    xB, yB = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, xB-xA) * max(0, yB-yA)
    areaA = max(0, a[2]-a[0]) * max(0, a[3]-a[1])
    areaB = max(0, b[2]-b[0]) * max(0, b[3]-b[1])
    return inter / (areaA + areaB - inter + 1e-6)

def ema(prev, new, alpha=EMA_ALPHA):
    return new if prev is None else (alpha*new + (1-alpha)*prev)

MODEL_3D = np.array([
    [-43.3,  32.7, -26.0],
    [ 43.3,  32.7, -26.0],
    [  0.0,   0.0,   0.0],
    [-28.9, -28.9, -24.1],
    [ 28.9, -28.9, -24.1],
    [  0.0, -63.6, -12.5],
], dtype=np.float64)

def estimate_head_pose_flexible(w, h, lm2d, lm3d_or_none):
    pts2d = np.array([[lm2d[i].x*w, lm2d[i].y*h] for i in LMK_IDX], dtype=np.float64)
    if lm3d_or_none is not None:
        pts3d = np.array([[lm3d_or_none[i].x, lm3d_or_none[i].y, lm3d_or_none[i].z] for i in LMK_IDX], dtype=np.float64)
    else:
        pts3d = MODEL_3D.copy()
    K = np.array([[w,0,w/2],[0,w,h/2],[0,0,1]], dtype=np.float64)
    dist = np.zeros((4,1))
    ok, rvec, tvec = cv2.solvePnP(pts3d, pts2d, K, dist, flags=cv2.SOLVEPNP_ITERATIVE)
    if not ok: return None, None, None
    R, _ = cv2.Rodrigues(rvec)
    sy = np.sqrt(R[0,0]**2 + R[1,0]**2)
    singular = sy < 1e-6
    if not singular:
        pitch = np.degrees(np.arctan2(R[2,1], R[2,2]))
        yaw   = np.degrees(np.arctan2(-R[2,0], sy))
        roll  = np.degrees(np.arctan2(R[1,0], R[0,0]))
    else:
        pitch = np.degrees(np.arctan2(-R[1,2], R[1,1])); yaw = np.degrees(np.arctan2(-R[2,0], sy)); roll = 0.0
    return yaw, pitch, roll

def gaze_ratio_h(w, h, lm2d):
    def eye_ratio(cout, cin, iris_idx):
        xo, xi = lm2d[cout].x*w, lm2d[cin].x*w
        xs = [lm2d[i].x*w for i in iris_idx if i < len(lm2d)]
        if not xs: return None
        xc = sum(xs)/len(xs)
        left, right = min(xo, xi), max(xo, xi)
        return (xc - left) / max(1.0, (right - left))
    rL = eye_ratio(EYE_LEFT_CORNERS[0], EYE_LEFT_CORNERS[1], GAZE_LEFT_LMK)
    rR = eye_ratio(EYE_RIGHT_CORNERS[0], EYE_RIGHT_CORNERS[1], GAZE_RIGHT_LMK)
    if rR is None:
        rR = eye_ratio(EYE_RIGHT_CORNERS[0], EYE_RIGHT_CORNERS[1], GAZE_RIGHT_LMK_VARIANT_B)
    if rL is not None and rR is not None: return 0.5*(rL+rR)
    return rL if rL is not None else rR

In [4]:
def iou(boxA, boxB):
    xA = max(boxA[0], boxB[0]); yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2]); yB = min(boxA[3], boxB[3])
    inter = max(0, xB-xA) * max(0, yB-yA)
    areaA = (boxA[2]-boxA[0]) * (boxA[3]-boxA[1])
    areaB = (boxB[2]-boxB[0]) * (boxB[3]-boxB[1])
    union = areaA + areaB - inter + 1e-6
    return inter / union

def ema(prev, new, alpha=EMA_ALPHA):
    if prev is None: return new
    return alpha * new + (1 - alpha) * prev

def rvec_tvec_to_euler(rvec):
    R, _ = cv2.Rodrigues(rvec)

    sy = np.sqrt(R[0, 0]**2 + R[1, 0]**2)
    singular = sy < 1e-6

    if not singular:
        pitch = np.degrees(np.arctan2(R[2, 1], R[2, 2]))
        yaw = np.degrees(np.arctan2(-R[2, 0], sy))
        roll = np.degrees(np.arctan2(R[1, 0], R[0, 0]))
    else:
        pitch = np.degrees(np.arctan2(-R[1, 2], R[1, 1]))
        yaw = np.degrees(np.arctan2(-R[2, 0], sy))
        roll = 0
    return yaw, pitch, roll

def now_ts():
    return time.time()

In [5]:
class Calib:
    def __init__(self):
        self.active = True
        self.t0 = None
        self.yaw_list, self.pitch_list, self.gaze_list = [], [], []
        self.bias_yaw, self.bias_pitch = 0.0, 0.0
        self.gaze_center = 0.5
        
    def start(self):
        self.active = True; self.t0 = now_ts()
        self.yaw_list.clear(); self.pitch_list.clear(); self.gaze_list.clear()

    def feed(self, yaw, pitch, g_h):
        if yaw is not None and pitch is not None:
            self.yaw_list.append(yaw); self.pitch_list.append(pitch)
        if g_h is not None: self.gaze_list.append(g_h)

    def done_if_ready(self):
        if not self.active or (now_ts() - self.t0) < CALIB_SECONDS: return False
        if self.yaw_list:
            self.bias_yaw = float(np.median(self.yaw_list))
        if self.pitch_list:
            self.bias_pitch = float(np.median(self.pitch_list))
        if self.gaze_list:
            self.gaze_center = float(np.median(self.gaze_list))
        self.active = False
        return True

class Track:
    _next_id = 1
    def __init__(self, bbox):
        self.id = Track._next_id; Track._next_id += 1
        self.bbox = bbox
        self.ts_last = now_ts()
        self.yaw_s = None; self.pitch_s = None; self.roll_s = None
        self.facing_prob = None
        self.state = "UNKNOWN"
        self._up_count = 0; self._down_count = 0
        self.ts_not_focus_start = None
        self.ts_eyes_off_start = None
        self.cheat_active = False
        self.cheat_reason = None
        self.ts_cheat_last = None

    def update_pose(self, yaw, pitch, roll):
        self.yaw_s = ema(self.yaw_s, yaw)
        self.pitch_s = ema(self.pitch_s, pitch)
        self.roll_s = ema(self.roll_s, roll)

    def update_bbox(self, bbox):
        self.bbox = bbox; self.ts_last = now_ts()

    def update_facing(self, facing_now):
        p = 1.0 if facing_now else 0.0
        self.facing_prob = ema(self.facing_prob, p)
        if facing_now:
            self._up_count += 1; self._down_count = 0
        else:
            self._down_count += 1; self._up_count = 0
        if self.state in ("UNKNOWN","NOT_FOCUS"):
            if self._up_count >= HYSTERESIS_FRAMES and (self.facing_prob or 0) >= 0.6:
                self.state = "FOCUS"
        if self.state in ("UNKNOWN","FOCUS"):
            if self._down_count >= HYSTERESIS_FRAMES and (self.facing_prob or 1) <= 0.4:
                self.state = "NOT_FOCUS"
        t = now_ts()
        if self.state == "NOT_FOCUS":
            if self.ts_not_focus_start is None: self.ts_not_focus_start = t
        else:
            self.ts_not_focus_start = None
            self.cheat_active = False; self.cheat_reason = None

    def update_gaze_flag(self, eyes_off):
        t = now_ts()
        if eyes_off:
            if self.ts_eyes_off_start is None: self.ts_eyes_off_start = t
        else:
            self.ts_eyes_off_start = None

    def mark_cheat(self, reason):
        self.cheat_active = True; self.cheat_reason = reason; self.ts_cheat_last = now_ts()

class Tracker:
    def __init__(self): self.tracks = []

    def update(self, bboxes):
        assigned = set()
        for t in self.tracks:
            best, idx = 0.0, -1
            for j, b in enumerate(bboxes):
                if j in assigned: continue
                i = iou(t.bbox, b)
                if i > best: best, idx = i, j
            if best >= IOU_THRESH and idx >= 0:
                t.update_bbox(bboxes[idx]); assigned.add(idx)
        for j, b in enumerate(bboxes):
            if j not in assigned: self.tracks.append(Track(b))
        tnow = now_ts()
        self.tracks = [t for t in self.tracks if (tnow - t.ts_last) <= MAX_MISSED_SEC]
        return self.tracks


In [6]:
def init_csv():
    if not ENABLE_CSV_LOG: return
    new = not os.path.exists(CSV_PATH)
    with open(CSV_PATH, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if new: w.writerow(["ts","track_id","state","yaw","pitch","roll","facing_prob","cheat","reason"])
        
def log_csv(t: Track):
    if not ENABLE_CSV_LOG: return
    with open(CSV_PATH, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow([f"{now_ts():.3f}", t.id, t.state,
                    f"{(t.yaw_s or 0):.2f}", f"{(t.pitch_s or 0):.2f}", f"{(t.roll_s or 0):.2f}",
                    f"{(t.facing_prob or 0):.2f}", int(t.cheat_active), t.cheat_reason or ""])

In [11]:
def main():
    init_csv()
    cap = cv2.VideoCapture(0)
    last_proc = 0.0
    tracker = Tracker()
    cal = Calib(); cal.start()

    mp_fd = mp.solutions.face_detection
    mp_fm = mp.solutions.face_mesh

    with mp_fd.FaceDetection(model_selection=0, min_detection_confidence=0.5) as fd, \
         mp_fm.FaceMesh(max_num_faces=5, refine_landmarks=True,
                        min_detection_confidence=0.5, min_tracking_confidence=0.5) as fm:

        while True:
            ok, frame = cap.read()
            if not ok: break
            h, w = frame.shape[:2]
            draw = frame.copy()
            tnow = now_ts()
            do_process = (tnow - last_proc) >= PROCESS_INTERVAL

            if do_process:
                last_proc = tnow
                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

                bboxes = []
                det = fd.process(rgb)
                if det and det.detections:
                    for d in det.detections:
                        rel = d.location_data.relative_bounding_box
                        x1 = int(rel.xmin*w); y1 = int(rel.ymin*h)
                        x2 = int((rel.xmin+rel.width)*w); y2 = int((rel.ymin+rel.height)*h)
                        x1 = max(0,x1); y1 = max(0,y1); x2 = min(w-1,x2); y2 = min(h-1,y2)
                        if x2>x1 and y2>y1: bboxes.append([x1,y1,x2,y2])

                tracks = tracker.update(bboxes)
                mesh_res = fm.process(rgb)

                lmk2d_list = []
                lmk3d_list = None
                if mesh_res and mesh_res.multi_face_landmarks:
                    lmk2d_list = mesh_res.multi_face_landmarks
                    has_world = hasattr(mesh_res,"multi_face_world_landmarks") and (mesh_res.multi_face_world_landmarks is not None)
                    lmk3d_list = mesh_res.multi_face_world_landmarks if has_world else [None]*len(lmk2d_list)

                    centers = []
                    for lm2d in lmk2d_list:
                        nose = lm2d.landmark[1]
                        centers.append((nose.x*w, nose.y*h))

                    for t in tracks:
                        cx = (t.bbox[0]+t.bbox[2])/2.0
                        cy = (t.bbox[1]+t.bbox[3])/2.0
                        if not centers: continue
                        idx = int(np.argmin([(cx-x)**2 + (cy-y)**2 for (x,y) in centers]))
                        lm2d = lmk2d_list[idx].landmark
                        lm3d = None if (lmk3d_list[idx] is None) else lmk3d_list[idx].landmark

                        yaw, pitch, roll = estimate_head_pose_flexible(w, h, lm2d, lm3d)
                        g_h = gaze_ratio_h(w, h, lm2d)

                        yaw_p = None if (yaw is None) else (yaw - cal.bias_yaw)
                        pit_p = None if (pitch is None) else (pitch - cal.bias_pitch)

                        t.update_pose(yaw_p, pit_p, roll)

                        cal.feed(yaw, pitch, g_h)
                        if cal.done_if_ready():
                            print(f"[CALIBRATED] bias_yaw={cal.bias_yaw:.1f}, bias_pitch={cal.bias_pitch:.1f}, gaze_center={cal.gaze_center:.2f}")

                        facing_now = (t.yaw_s is not None and t.pitch_s is not None and
                                      abs(t.yaw_s) <= FACING_YAW_DEG and abs(t.pitch_s) <= FACING_PITCH_DEG)
                        t.update_facing(facing_now)

                        eyes_off = False
                        if g_h is not None and not cal.active:
                            g_dev = abs(g_h - cal.gaze_center)
                            if facing_now and (abs(t.yaw_s) <= MIN_FACING_FOR_GAZE and abs(t.pitch_s) <= MIN_FACING_FOR_GAZE):
                                eyes_off = g_dev > GAZE_DELTA
                        t.update_gaze_flag(eyes_off)

                        if not cal.active:
                            nowt = now_ts()
                            if t.ts_not_focus_start is not None and (nowt - t.ts_not_focus_start) >= CHEAT_MIN_OFFPOSE_SEC:
                                t.mark_cheat("HEAD_POSE_OFF"); log_csv(t)
                            if t.ts_eyes_off_start is not None and (nowt - t.ts_eyes_off_start) >= CHEAT_MIN_EYESOFF_SEC:
                                t.mark_cheat("EYES_OFF"); log_csv(t)

                if len(tracker.tracks) > 1 and not cal.active:
                    oldest_seen = min(t.ts_last for t in tracker.tracks)
                    if (tnow - oldest_seen) >= CHEAT_MIN_MULTIFACE_SEC:
                        main_t = max(tracker.tracks, key=lambda tr: tr.ts_last)
                        main_t.mark_cheat("MULTIPLE_FACES"); log_csv(main_t)

                for t in tracker.tracks:
                    if (tnow - t.ts_last) >= CHEAT_MIN_OUTOFFRAME_SEC and not cal.active:
                        t.mark_cheat("OUT_OF_FRAME"); log_csv(t)

            for t in tracker.tracks:
                x1,y1,x2,y2 = map(int, t.bbox)
                base = (0,200,0) if t.state=="FOCUS" else (0,0,200)
                color = (0,0,255) if t.cheat_active else base
                cv2.rectangle(draw, (x1,y1), (x2,y2), color, 2)
                label = f"ID {t.id} | {t.state}"
                
                if t.yaw_s is not None and t.pitch_s is not None:
                    label += f" | yaw {t.yaw_s:+.0f}° pitch {t.pitch_s:+.0f}°"  # turn on if needed

                if t.cheat_active and t.cheat_reason:
                    label += f" | CHEATING:{t.cheat_reason}"
                cv2.putText(draw, label, (x1, max(20, y1-8)), cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)

            if len(tracker.tracks) > 1:
                cv2.putText(draw, f"MULTIPLE FACES: {len(tracker.tracks)}", (10, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,255), 2)

            if SHOW_DEBUG:
                cv2.putText(draw, f"thr yaw<=±{int(FACING_YAW_DEG)}, pitch<=±{int(FACING_PITCH_DEG)}", (10, 50),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200,200,200), 1)
                cv2.putText(draw, f"Proc ~{TARGET_FPS} FPS | Tracks:{len(tracker.tracks)}", (10, h-10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)
                if 'cal' in locals():
                    dbg = f"bias(y,p)=({cal.bias_yaw:+.1f},{cal.bias_pitch:+.1f}) center={cal.gaze_center:.2f} d={GAZE_DELTA:.2f}"
                    cv2.putText(draw, dbg, (10, 70), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (200,200,200), 1)

            cv2.imshow("Cheating Detection (Head Pose + Gaze, Calibrated)", draw)
            if cv2.waitKey(1) & 0xFF == 27: break

    cap.release(); cv2.destroyAllWindows()


In [12]:
main()

[CALIBRATED] bias_yaw=-20.3, bias_pitch=177.4, gaze_center=0.20
